# 04_score_and_segment

**Owner:** Models and evaluation. **Blueprint steps:** 4 and 9. **Status:** Runs locally. Not yet run in Fabric.

Scores accounts with pinned model versions, assigns segments and writes Gold tables. Scoring never retrains.

1. Write a `score_runs` row with status `running`.
2. Check the snapshots (INV-2, feature version, run cutoff).
3. Append `account_scores`, `feature_attributions` and `segment_membership`. Segment rows are never overwritten.
4. Write one `risk_outbox` row per account with a stable `event_id`.
5. Mark the run `completed`. Completion is the visibility gate.
6. Publish `risk_updated` from the outbox (the publish step of `pl_scheduled_scoring`).

INV-5 / T05: a failed run is marked `failed`. It writes no outbox rows and publishes no `risk_updated` event. The last completed score shows as stale. Set `INJECT_FAILURE` to demonstrate this.

Run kinds: `checkpoint` scores the checkpoint snapshots of open deal cycles. `scheduled` and `manual` score the live snapshots that `02_feature_snapshots` wrote at `CUTOFF`.

In [ ]:
OUTPUT_ROOT = None  # default: data/output
RUN_KIND = 'checkpoint'  # 'checkpoint', 'scheduled' or 'manual'
CUTOFF = None  # ISO 8601. Default: the generator as_of time
STARTED_AT = None  # simulated run start (ISO 8601). Default: now
MODEL_VERSION = None  # pinned classifier, for example 'hb-refi-clf:1'. Default: latest release
CLUSTER_MODEL_VERSION = None  # pinned cluster model. Default: same release
MORTGAGE_IDS = None  # optional list of mortgage_ids
BUILD_LIVE_SNAPSHOTS = False  # local convenience: build live snapshots here instead of in 02_feature_snapshots
INJECT_FAILURE = None  # T05 demo: 'before_outputs' or 'after_outputs'
PUBLISH = True  # publish risk_updated after a completed run
PUBLISH_ONLY = False  # retry pending publication without scoring

In [ ]:
import sys
from pathlib import Path

# Local run: this notebook sits in notebooks/ and imports notebooks/models_eval.
# Fabric run (not yet done): upload models_eval as a workspace resource or environment library,
# set HB_OUTPUT_ROOT to the Lakehouse Files path and HB_MLFLOW_URI = 'fabric'.
NOTEBOOKS_DIR = next(p for p in [Path.cwd(), Path.cwd() / 'notebooks'] if (p / 'models_eval').exists())
sys.path.insert(0, str(NOTEBOOKS_DIR))
import json
import pandas as pd
from models_eval import config
PATHS = config.paths(OUTPUT_ROOT)
print('Output root:', PATHS.root)

In [ ]:
from models_eval import pipeline, publish, scoring
result = None
if not PUBLISH_ONLY:
    if CUTOFF is None:  # local default: the generator as_of time
        manifest = json.loads((PATHS.root / 'exports' / 'manifest.json').read_text(encoding='utf-8'))
        CUTOFF = manifest['config']['as_of']
    cutoff = pd.Timestamp(CUTOFF)
    if BUILD_LIVE_SNAPSHOTS and RUN_KIND != 'checkpoint':
        pipeline.live_snapshots(PATHS, cutoff, MORTGAGE_IDS)
    snap = pipeline.select_snapshots(PATHS, cutoff, RUN_KIND, MORTGAGE_IDS)
    print(len(snap), 'snapshots at cutoff', cutoff)
    # A failure raises ScoringRunFailed. The pipeline activity then fails and the publish step does not run.
    result = scoring.run_scoring(PATHS, snap, RUN_KIND, cutoff, MODEL_VERSION, CLUSTER_MODEL_VERSION,
                                 started_at=STARTED_AT, inject_failure=INJECT_FAILURE)
result

In [ ]:
publication = publish.publish_pending(PATHS, now=STARTED_AT) if (PUBLISH or PUBLISH_ONLY) else 'skipped'
publication

## Run status

Only completed runs are visible to the report and the case writer.

In [ ]:
from models_eval import gold
gold.read(PATHS.gold, 'score_runs').sort_values('started_at').tail(5)

## Latest score per account

The Data and replay `score_context` rule: a score is stale when a later run failed (T05) or the score is old.

In [ ]:
as_of = pd.Timestamp(STARTED_AT) if STARTED_AT else pd.Timestamp.now(tz='UTC')
latest = scoring.latest_scores(PATHS, as_of)
latest['score_state'].value_counts()

## Ten highest scores in this run

The proposed capacity is ten reviews per run. Attributions explain the score. They do not show the borrower's motive. Payment-support routing (INV-1) does not depend on these scores.

In [ ]:
if result:
    scores = gold.read(PATHS.gold, 'account_scores')
    top = scores[scores['run_id'] == result['run_id']].nlargest(10, 'score')
    attr = gold.read(PATHS.gold, 'feature_attributions')
    attr = attr[(attr['run_id'] == result['run_id']) & (attr['rank'] <= 3)]
    drivers = attr.groupby('account_key').apply(lambda d: '; '.join(f"{f} ({c:+.2f})" for f, c in zip(d['feature_name'], d['contribution'])), include_groups=False)
    display(top[['account_key', 'score', 'segment_id', 'snapshot_id']].assign(top_drivers=top['account_key'].map(drivers)))

## Segment counts for this run

Each run appends one row per account. Earlier rows never change.

In [ ]:
if result:
    seg = gold.read(PATHS.gold, 'segment_membership')
    display(seg[seg['run_id'] == result['run_id']]['segment_id'].value_counts().sort_index())